# RF-DETR 1.6.0: From Quick Start to Full Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/notebooks/release-demo_1-6.ipynb)

In 1.6.0, the training stack is built on PyTorch Lightning building blocks
that you can compose however you like — but you don't have to use them directly.

| Building block | Role |
|---|---|
| `RFDETRModelModule` | `LightningModule` — model, loss, optimizer, scheduler |
| `RFDETRDataModule` | `LightningDataModule` — datasets and dataloaders |
| `build_trainer()` | Factory that assembles `Trainer` with all RF-DETR callbacks |

This notebook demonstrates the key design principle: **start simple, then pick up
building blocks without losing your trained weights**.

- **Phase 1** — `model.train()` one-liner, 5 epochs
- **Phase 2** — swap in the PTL components and continue for 10 more epochs from the
  same checkpoint, same output folder — no conversion required
- **End** — full training curve, then inference

## 1. Install RF-DETR 1.6.0

Install directly from the tagged release archive on GitHub.

In [ ]:
!pip install -q "rfdetr[train,loggers]==1.6.0" roboflow

## 2. Config

In [ ]:
import os
from pathlib import Path

DATASET_DIR = os.environ.get("DATASET_DIR", "")
OUTPUT_DIR = "output"
EPOCHS_PHASE_1 = 2
EPOCHS_PHASE_2 = 3
BATCH_SIZE = 12

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Workers

In [ ]:
try:
    from IPython import get_ipython

    _in_notebook = get_ipython() is not None
except Exception:
    _in_notebook = False

# Outside a notebook kernel, macOS/Windows spawn-based multiprocessing will
# re-import this script as __main__, triggering training again.
# Use 0 workers in that case; inside a kernel the usual forking rules apply safely.
num_workers = os.cpu_count() if _in_notebook else 0

## 4. Dataset

[Aquarium](https://universe.roboflow.com/brad-dwyer/aquarium-combined) — 638 images, 7 classes.

In [ ]:
import json

from roboflow import Roboflow

try:
    from google.colab import userdata  # type: ignore[import]

    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = os.environ["ROBOFLOW_API_KEY"]

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace("brad-dwyer").project("aquarium-combined").version(1).download("coco", location="datasets")
DATASET_DIR = dataset.location

with open(Path(DATASET_DIR) / "train" / "_annotations.coco.json") as f:
    _ann = json.load(f)

CLASS_NAMES = [c["name"] for c in sorted(_ann["categories"], key=lambda c: c["id"])]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Dataset : {DATASET_DIR}")
print(f"Classes : {NUM_CLASSES} — {CLASS_NAMES}")

## 5. Phase 1 — `model.train()` (5 epochs)

The familiar one-liner API. Nothing changes from previous releases.
After training, `OUTPUT_DIR/checkpoint_best_total.pth` holds the best weights.

In [ ]:
from rfdetr import RFDETRMedium

model = RFDETRMedium(num_classes=NUM_CLASSES, pretrain_weights="rf-detr-medium.pth")
model.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS_PHASE_1,
    batch_size=BATCH_SIZE,
    grad_accum_steps=4,
    lr=1e-4,
    num_workers=num_workers,
    output_dir=OUTPUT_DIR,
    use_ema=True,
    run_test=False,
    progress_bar="rich",
    tensorboard=True,
    seed=42,
)

## 6. Phase 2 — PTL building blocks (10 more epochs)

Pick up the three PTL components and call `trainer.fit()` pointing at the Phase 1
checkpoint.  No conversion step — `RFDETRModelModule.on_load_checkpoint` detects and
handles the `.pth` format automatically.

A lower learning rate (`5e-5`) is used because the model is already partially
converged.  The same `OUTPUT_DIR` is reused so checkpoints and metrics land in
one place.

In [ ]:
import pandas as pd

from rfdetr import RFDETRDataModule, RFDETRModelModule, build_trainer
from rfdetr.config import RFDETRMediumConfig, TrainConfig

# Read Phase 1 metrics before Phase 2 overwrites the CSV.
df1 = pd.read_csv(f"{OUTPUT_DIR}/metrics.csv")

model_config = RFDETRMediumConfig(
    num_classes=NUM_CLASSES,
    pretrain_weights="rf-detr-medium.pth",
)

# epochs = EPOCHS_1 + EPOCHS_2 so PTL (which resumes the epoch counter from the
# checkpoint) runs exactly EPOCHS_2 additional epochs before reaching max_epochs.
train_config = TrainConfig(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS_PHASE_1 + EPOCHS_PHASE_2,
    batch_size=BATCH_SIZE,
    grad_accum_steps=4,
    lr=5e-5,
    num_workers=num_workers,
    output_dir=OUTPUT_DIR,
    use_ema=True,
    run_test=True,
    progress_bar="tqdm",
    tensorboard=True,
    seed=42,
)

module = RFDETRModelModule(model_config=model_config, train_config=train_config)
datamodule = RFDETRDataModule(model_config=model_config, train_config=train_config)
trainer = build_trainer(train_config, model_config)

# Resume directly from the Phase 1 .pth — no conversion needed.
trainer.fit(module, datamodule, ckpt_path=f"{OUTPUT_DIR}/checkpoint_best_total.pth")

## 7. Training curve

Concatenate the Phase 1 metrics we saved earlier with the Phase 2 metrics written
by PTL, then plot a single four-panel chart across all epochs.

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display

from rfdetr.visualize.training import plot_metrics

df2 = pd.read_csv(f"{OUTPUT_DIR}/metrics.csv")

combined_csv = f"{OUTPUT_DIR}/metrics_combined.csv"
pd.concat([df1, df2], ignore_index=True).to_csv(combined_csv, index=False)
print(f"Combined CSV: {combined_csv}  ({len(df1) + len(df2)} rows)")

plot_path = plot_metrics(combined_csv)
display(IPyImage(plot_path))
print(f"Saved: {plot_path}")

## 8. Inference

Load the best checkpoint and run on a validation image.

In [ ]:
import supervision as sv
from PIL import Image

model = RFDETRMedium(pretrain_weights=f"{OUTPUT_DIR}/checkpoint_best_total.pth", num_classes=NUM_CLASSES)

val_ann = Path(DATASET_DIR) / "valid" / "_annotations.coco.json"
with open(val_ann) as f:
    ann_data = json.load(f)

image = Image.open(Path(DATASET_DIR) / "valid" / ann_data["images"][0]["file_name"])
detections = model.predict(image, threshold=0.3)

annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
annotated = sv.LabelAnnotator().annotate(annotated, detections, labels=[CLASS_NAMES[c] for c in detections.class_id])
sv.plot_image(annotated)
print(f"Detected {len(detections)} object(s)")

## 9. Next steps

- [PyTorch Lightning training docs](https://rfdetr.roboflow.com/develop/learn/train/pytorch-lightning/)
- [Advanced training options](https://rfdetr.roboflow.com/develop/learn/train/advanced/)
- [Logger integrations (ClearML, MLflow, W&B)](https://rfdetr.roboflow.com/develop/learn/train/loggers/)
- [Export your model](https://rfdetr.roboflow.com/develop/learn/export/)